# Cellpose segmentation

We will use the [Cellpose](https://www.cellpose.org/) model to segment the images. Cellpose is a generalist algorithm for cellular segmentation that can be applied to a wide range of cell types and imaging modalities.

In [ ]:
# Needs to be installed before running the notebook
!pip install git+https://github.com/FLClab/TiffWrapper.git

In [7]:
import random
import cellpose
import numpy
import pandas
import matplotlib

from cellpose import models, io, utils
from matplotlib import pyplot
from bioio import BioImage
import bioio_bioformats
from tiffwrapper import make_composite

# Create the Cellpose model instance with GPU enabled or disabled
model = models.CellposeModel(gpu=False)

Some utility functions are provided to help with the segmentation process, including functions to shuffle the masks for better visualization and to display the original image alongside the segmentation results.

In [ ]:
def shuffle_masks(masks):
    """
    Shuffle the labels in the masks array to randomize the label values.

    Parameters:
    -----------
    masks : numpy.ndarray
        The input masks array with labeled regions.

    Returns:
    --------
    numpy.ndarray
        The shuffled masks array with randomized label values.
    """
    unique_labels = numpy.unique(masks)[1:] # Exclude the background label (0)
    shuffled_labels = numpy.random.permutation(unique_labels)
    label_mapping = {old: new for old, new in zip(unique_labels, shuffled_labels)}
    
    shuffled_masks = numpy.copy(masks)
    for old_label, new_label in label_mapping.items():
        shuffled_masks[masks == old_label] = new_label
    
    return shuffled_masks

In [ ]:
# Create a list of images to segment
import os
import glob

def list_images(root: str) -> list:
    """
    List all image files in the specified root directory.

    Parameters:
    -----------
    root : str
        The root directory to search for image files.

    Returns:
    --------
    list
        A list of file paths to the image files found in the root directory.
    """
    # TODO - Hint: Use glob.glob to find all image files in the root directory with extensions *.oir
    image_paths = []
    return image_paths

image_paths = list_images('/path/to/your/images')

assert len(image_paths) > 0, "No images found in the specified directory. Please check the path and try again."

In [ ]:
# Create a function that reads an image and returns the data
def read_image(image_path: str) -> numpy.ndarray:
    """
    Read an image file and return the image data as a numpy array.

    Parameters:
    -----------
    image_path : str
        The path to the image file to be read.

    Returns:
    --------
    numpy.ndarray
        The image data as a numpy array.
    """
    # TODO - 1: Use BioImage to read the image and return the data as a numpy array. Make sure to use the `reader=bioio_bioformats.Reader` option
    #        2: Use the .data attribute of the BioImage object to access the image data
    #        3: BioImage returns a 5D array (t, c, z, y, x), we need to select the first time point and the first z-slice to get a 3D array (c, y, x)

    img = None
    return img

img = read_image(image_paths[0])

assert img.ndim == 3, "The image data should be a 3D array with shape (c, y, x)."
assert img.shape[0] == 4, "The image data should have 4 channels (c=4)."

fig, ax = pyplot.subplots(1, 1, figsize=(10, 10))
composite = make_composite(img, luts=["blue", "red", "green", "magenta"])
ax.imshow(composite)
ax.axis("off")
pyplot.show()

In the next cell, we will use the Cellpose model to segment the images. We will set the parameters for the segmentation process, including the object diameter, flow threshold, and cell probability threshold. These parameters can be adjusted to optimize the segmentation results for your specific images.

Running this cell assumes that you have already implemented the `list_images` and `read_image` functions in the cells above. If you have not done so, please implement those functions before running the next cell.

In [ ]:
# Set parameters for Cellpose segmentation
# TODO: Change the obj_diameter, flow_threshold, and cellprob_threshold values to optimize segmentation results for your specific images.
obj_diameter = 40 # in pixels
flow_threshold = 0.4
cellprob_threshold = 0.0

# Images contain four channels, we must set the channel_to_segment variable to the index of the channel we want to segment.
channel_to_segment = 3  # 0 for DAPI, 1 for CaMKII, 2 for GFP, 3 for NeuN

# Randomly select an image from the list of images to segment
image_path = random.choice(image_paths)

# data should be a 3D array with shape (c, y, x) where c is the number of channels, y is the height and x is the width of the image
data = read_image(image_path)

# Resize image to 256x256 to reduce computation time and memory usage
data = cellpose.transforms.resize_image(data, 256, 256, no_channels=True)

# Run Cellpose segmentation
masks, flows, styles = model.eval(
    data, 
    diameter=obj_diameter / (data.shape[1] / 256),  # Scale diameter according to image size
    channels=[channel_to_segment, channel_to_segment], 
    do_3D=False, 
    flow_threshold=flow_threshold, 
    cellprob_threshold=cellprob_threshold
)

# Display image and segmentation
fig, axes = pyplot.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(data[channel_to_segment], cmap='gray')
axes[1].imshow(shuffle_masks(masks), cmap="magma")
for ax in axes:
    ax.axis('off')
pyplot.show()

Now that we found the parameters that work best for our images, we can run the Cellpose model on all the images in the dataset. The following code will iterate through each image, read it, and apply the Cellpose model to segment the cells. The results will be stored in a list named `results` for further analysis or visualization.

In [ ]:
# TODO: Write a for loop that iterates through all the images in the image_paths list, reads each image, and applies the Cellpose model to segment the cells. 
# Store the results in a list for further analysis or visualization.

results = []
for ... :
    results.append({
        "image" : data,
        "masks" : masks,
    })

We will now attempt to split the cells that were segmented in the images into different groups. In the dataset, the cells have different types (excitatory and inhibitory), and we will use the segmentation results to separate them into different groups. This will allow us to analyze the properties of each cell type separately. We will attempt to do that automatically using the segmentation results and the original images.

In [ ]:
from skimage import measure
# Since we need to get the properties from multiple channels at the same time
intensity_channels = [1, 2]

image = results[0]["image"]
masks = results[0]["masks"]

# TODO: Use skimage.measure.regionprops to extract features from the segmented masks and the intensity images. 
#       Only index the channels that you want to extract features from in the intensity_image parameter.
#       Hint: the intensity_image can be a 3D array with shape (y, x, c) where c is the number of channels. 
#             You can use numpy.transpose to rearrange the dimensions of the data array to match this shape.
rprops = measure.regionprops(...)

# Extract features from the region properties and store them in a list of dictionaries
features = []
for prop in rprops:
    feature = {
        "label" : prop.label,
        "mean_intensity_CaMKII": prop.intensity_mean[0],
        "mean_intensity_EGFP": prop.intensity_mean[1],
        # Add more features as needed
    }
    features.append(feature)

In the next cell, we will now create a `pandas.DataFrame` from the extracted features for easier analysis. The DataFrame will contain the mean intensities of the different channels for each segmented cell, allowing us to analyze the properties of each cell type separately.

In [ ]:
# TODO: Convert the list of feature dictionaries to a pandas DataFrame for easier analysis
features_df = ...

assert isinstance(features_df, pandas.DataFrame), "features_df should be a pandas DataFrame."
assert "mean_intensity_CaMKII" in features_df.columns and "mean_intensity_EGFP" in features_df.columns, "features_df should have the correct columns."

print(features_df.head())

While printing the DataFrame can be useful for debugging, it only provides a limited view of the data. To gain a better understanding of the distribution of the features, we will create histograms for each feature in the DataFrame. This will allow us to visualize the distribution of the mean intensities for each channel and identify any patterns or differences between the cell types.

Do you see any patterns or differences between the cell types based on their mean intensities in the different channels?

In [ ]:
# TODO: Plot the distribution of mean intensities for each channel
#       Hint: Use the `ax.hist` function to create the historgram of the data, make sure to specify the labels.
for key in features_df.keys():
    fig, ax = pyplot.subplots(figsize=(4,3))
    ...
    pyplot.show()

Another way to visualize the data is to plot the relationship between the different channels. This gives a better overview of the data and allows us to see if there are any correlations between the channels. We will create a scatter plot to visualize the relationship between the mean intensities of the two channels for each segmented cell. This will help us identify any patterns or differences between the cell types based on their mean intensities in the different channels.

Do you see any patterns or differences between the cell types based on their mean intensities in the different channels?

In [ ]:
# TODO: Plot the relationship between mean intensities of the two channels using the `ax.scatter` function
fig, ax = pyplot.subplots(figsize=(4,3))
...
pyplot.show()

We will now attempt to split the cells that were segmented in the images into different groups. In the dataset, the cells have different types (excitatory and inhibitory), we will use their intensity features to separate them into different groups. This will allow us to analyze the properties of each cell type separately. We will do that automatically by using a KMeans clustering algorithm on the extracted features. The KMeans algorithm will group the cells based on their mean intensities in the different channels, allowing us to identify distinct cell types in the dataset.

In [ ]:
# Attempt to cluster the features using KMeans
from sklearn.cluster import KMeans

n_clusters = 6  # TODO: Change this value to the number of clusters you want to use for KMeans clustering
kmeans = ...
cluster_labels = ...

fig, ax = pyplot.subplots(figsize=(4,3))
scatter = ax.scatter(features_df["mean_intensity_CaMKII"], features_df["mean_intensity_EGFP"], c=cluster_labels, cmap='viridis', alpha=0.7)
legend1 = ax.legend(*scatter.legend_elements(), title="Clusters")
ax.add_artist(legend1)
ax.set_title('KMeans Clustering of Mean Intensities')
ax.set_xlabel('Mean Intensity CaMKII')
ax.set_ylabel('Mean Intensity EGFP')
pyplot.show()

In the previous cell, we used KMeans clustering to group the cells based on their mean intensities in the different channels. The scatter plot shows the relationship between the mean intensities of the two channels, with each point representing a segmented cell and colored according to its assigned cluster.

We will now attempt to visualize the cell type of the segmented cells in the original images. 

In [ ]:
# Extract the image and associated masks from the results for visualization
image = results[0]["image"]
masks = results[0]["masks"]

fig, axes = pyplot.subplots(1, 2, figsize=(10, 5))
composite = make_composite(image[intensity_channels], luts=["red", "green"], ranges=numpy.quantile(image[intensity_channels], q=[0., 0.99], axis=(1, 2)).T)
axes[0].imshow(composite)

# Create a new mask where each region is colored according to its cluster label
cluster_mask = numpy.zeros_like(masks)
for row_index, cell in features_df.iterrows():
    # TODO: Use the cluster_labels array to assign a cluster label to each cell in the cluster_mask.
    cluster_mask[...] = ...
axes[1].imshow(cluster_mask, cmap="magma")

cmap = pyplot.get_cmap("magma", best_n_clusters+1)
legend = axes[1].legend(
    handles=[matplotlib.patches.Patch(color=cmap(i+1), label=f"C{i}") for i in range(best_n_clusters)],
)
for ax in axes:
    ax.axis('off')
pyplot.show()